# Calculate CER and WER Metrics

This notebook calculates Character Error Rate (CER) and Word Error Rate (WER) from a file where each line has the format:

`predicted_text****ground_truth`

## Metrics:
- **CER (Character Error Rate)**: Edit distance at character level
- **WER (Word Error Rate)**: Edit distance at word level

Both are calculated using Levenshtein distance (insertions, deletions, substitutions).

In [1]:
import numpy as np
from typing import List, Tuple


In [2]:
# Configuration
input_file = "matched_annotations.txt"


In [3]:
def levenshtein_distance(seq1: List, seq2: List) -> int:
    """
    Calculate the Levenshtein distance between two sequences.
    
    Args:
        seq1: First sequence (list of characters or words)
        seq2: Second sequence (list of characters or words)
    
    Returns:
        Edit distance (number of insertions, deletions, substitutions)
    """
    size_x = len(seq1) + 1
    size_y = len(seq2) + 1
    
    # Create distance matrix
    matrix = np.zeros((size_x, size_y), dtype=int)
    
    # Initialize first column and row
    for x in range(size_x):
        matrix[x, 0] = x
    for y in range(size_y):
        matrix[0, y] = y
    
    # Fill the matrix
    for x in range(1, size_x):
        for y in range(1, size_y):
            if seq1[x-1] == seq2[y-1]:
                matrix[x, y] = matrix[x-1, y-1]
            else:
                matrix[x, y] = min(
                    matrix[x-1, y] + 1,      # deletion
                    matrix[x, y-1] + 1,      # insertion
                    matrix[x-1, y-1] + 1     # substitution
                )
    
    return matrix[size_x - 1, size_y - 1]


In [4]:
def calculate_cer(predicted: str, ground_truth: str) -> Tuple[int, int]:
    """
    Calculate Character Error Rate components.
    
    Args:
        predicted: Predicted text
        ground_truth: Ground truth text
    
    Returns:
        Tuple of (edit_distance, reference_length)
    """
    pred_chars = list(predicted)
    gt_chars = list(ground_truth)
    
    distance = levenshtein_distance(pred_chars, gt_chars)
    return distance, len(gt_chars)


def calculate_wer(predicted: str, ground_truth: str) -> Tuple[int, int]:
    """
    Calculate Word Error Rate components.
    
    Args:
        predicted: Predicted text
        ground_truth: Ground truth text
    
    Returns:
        Tuple of (edit_distance, reference_length)
    """
    pred_words = predicted.split()
    gt_words = ground_truth.split()
    
    distance = levenshtein_distance(pred_words, gt_words)
    return distance, len(gt_words)


In [5]:
# Test the functions with an example
print("Testing CER and WER functions:")
print("="*60)

test_pred = "hello world"
test_gt = "helo world"

cer_dist, cer_len = calculate_cer(test_pred, test_gt)
wer_dist, wer_len = calculate_wer(test_pred, test_gt)

print(f"Predicted: '{test_pred}'")
print(f"Ground Truth: '{test_gt}'")
print(f"\nCER: {cer_dist}/{cer_len} = {cer_dist/cer_len*100:.2f}%")
print(f"WER: {wer_dist}/{wer_len} = {wer_dist/wer_len*100:.2f}%")


Testing CER and WER functions:
Predicted: 'hello world'
Ground Truth: 'helo world'

CER: 1/10 = 10.00%
WER: 1/2 = 50.00%


In [6]:
# Read the input file and calculate overall CER and WER
print(f"\nReading {input_file}...\n")

total_cer_distance = 0
total_cer_length = 0
total_wer_distance = 0
total_wer_length = 0

line_count = 0
error_count = 0

with open(input_file, 'r', encoding='utf-8') as f:
    for line_num, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        
        # Split by '****' separator
        parts = line.split('****')
        
        if len(parts) != 2:
            print(f"Warning: Line {line_num} doesn't have correct format (missing '****')")
            error_count += 1
            continue
        
        predicted = parts[0].strip()
        ground_truth = parts[1].strip()
        
        # Calculate CER for this line
        cer_dist, cer_len = calculate_cer(predicted, ground_truth)
        total_cer_distance += cer_dist
        total_cer_length += cer_len
        
        # Calculate WER for this line
        wer_dist, wer_len = calculate_wer(predicted, ground_truth)
        total_wer_distance += wer_dist
        total_wer_length += wer_len
        
        line_count += 1
        
        # Show progress every 100 lines
        if line_count % 100 == 0:
            print(f"Processed {line_count} lines...")

print(f"\nFinished processing {line_count} lines")
if error_count > 0:
    print(f"Skipped {error_count} lines due to format errors")



Reading matched_annotations.txt...

Processed 100 lines...
Processed 200 lines...
Processed 300 lines...

Finished processing 307 lines


In [7]:
# Calculate overall CER and WER
if total_cer_length > 0:
    overall_cer = (total_cer_distance / total_cer_length) * 100
else:
    overall_cer = 0

if total_wer_length > 0:
    overall_wer = (total_wer_distance / total_wer_length) * 100
else:
    overall_wer = 0

# Display results
print("\n" + "="*60)
print("OVERALL METRICS")
print("="*60)
print(f"\nTotal lines processed: {line_count}")
print(f"\n{'Metric':<20} {'Distance':<15} {'Length':<15} {'Rate'}")
print("-"*60)
print(f"{'CER':<20} {total_cer_distance:<15} {total_cer_length:<15} {overall_cer:.4f}%")
print(f"{'WER':<20} {total_wer_distance:<15} {total_wer_length:<15} {overall_wer:.4f}%")
print("="*60)



OVERALL METRICS

Total lines processed: 307

Metric               Distance        Length          Rate
------------------------------------------------------------
CER                  4955            13447           36.8484%
WER                  1757            1785            98.4314%


In [8]:
# Show some example predictions with their errors
print("\nSample predictions (first 5 lines):")
print("="*80)

with open(input_file, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f, 1):
        if i > 5:
            break
        
        line = line.strip()
        if not line:
            continue
        
        parts = line.split('****')
        if len(parts) == 2:
            predicted = parts[0].strip()
            ground_truth = parts[1].strip()
            
            cer_dist, cer_len = calculate_cer(predicted, ground_truth)
            wer_dist, wer_len = calculate_wer(predicted, ground_truth)
            
            cer_val = (cer_dist / cer_len * 100) if cer_len > 0 else 0
            wer_val = (wer_dist / wer_len * 100) if wer_len > 0 else 0
            
            print(f"\nLine {i}:")
            print(f"  Predicted:    {predicted[:80]}{'...' if len(predicted) > 80 else ''}")
            print(f"  Ground Truth: {ground_truth[:80]}{'...' if len(ground_truth) > 80 else ''}")
            print(f"  CER: {cer_val:.2f}%  |  WER: {wer_val:.2f}%")



Sample predictions (first 5 lines):

Line 1:
  Predicted:    सोपुनीत सवपदल है दर्द चतुर्गतिवास। दच
  Ground Truth: सोपुनीतसिवपदल है।दहैचतुर्गतिवास।दहैचतुर्ग
  CER: 36.59%  |  WER: 300.00%

Line 2:
  Predicted:    3 | न 27 (1वগত । मान
  Ground Truth: डले: ॥ दर्शादगंपोमपपैनो शर्मध्व जसो मधेकृय: । २८ ॥ राजगच्छव त्रेक विपुली मैत्रीक...
  CER: 90.35%  |  WER: 94.74%

Line 3:
  Predicted:    इमदेय सामनजातीप्रदः ।
  Ground Truth: डुषदेय सोतुमनैजीतीप्रभुः ज
  CER: 42.31%  |  WER: 100.00%

Line 4:
  Predicted:    कैभरे भंडार||भ्यातंजीमोलोभनकोसाथ
  Ground Truth: कैभरे भंडार॥२२॥तंज दीयोलोभनकोसाथ
  CER: 28.12%  |  WER: 66.67%

Line 5:
  Predicted:    न अवधिज्ञान नैं जा रायौ भेद सिर धुन्यौ मनयायौ स्वेदा ५१ पुष्पदंत चाल्यौ अब सारा ...
  Ground Truth: न अवधिज्ञान तें जारायौ भेद सिरधुन्यौ मनपायो खेद  ।५०। पुष्पदंत चाल्यो अबसार श्री...
  CER: 24.73%  |  WER: 86.67%


In [9]:
# Save results to a file
results_file = "cer_wer_results.txt"

with open(results_file, 'w', encoding='utf-8') as f:
    f.write("CER and WER Calculation Results\n")
    f.write("="*60 + "\n\n")
    f.write(f"Input file: {input_file}\n")
    f.write(f"Total lines processed: {line_count}\n\n")
    f.write(f"Character Error Rate (CER):\n")
    f.write(f"  Total character edit distance: {total_cer_distance}\n")
    f.write(f"  Total reference characters: {total_cer_length}\n")
    f.write(f"  Overall CER: {overall_cer:.4f}%\n\n")
    f.write(f"Word Error Rate (WER):\n")
    f.write(f"  Total word edit distance: {total_wer_distance}\n")
    f.write(f"  Total reference words: {total_wer_length}\n")
    f.write(f"  Overall WER: {overall_wer:.4f}%\n")

print(f"\nResults saved to: {results_file}")



Results saved to: cer_wer_results.txt
